# TP Búsqueda Voraz y A*

Notebook listo para ejecutar en Google Colab. Contiene el desarrollo completo del trabajo: definición del grafo, implementación de UCS, Voraz y A*, ejecución comparativa y guardado de resultados.

La idea es que puedas usarlo sin depender de archivos externos. Si querés, también podés adaptar la sección de Drive para cargar el repositorio desde Colab.

## Importar librerías y configuración inicial

Se usan únicamente bibliotecas estándar de Python para que el notebook sea portable y no dependa de instalaciones adicionales.

In [ ]:
import heapq
import itertools
import json
from pathlib import Path
from pprint import pprint

print('Librerías importadas correctamente.')

## Inspección inicial del conjunto de datos

En este trabajo el "conjunto de datos" es el grafo de búsqueda. Acá se inspeccionan nodos, aristas y heurísticas para entender el problema antes de ejecutar los algoritmos.

In [ ]:
grafo = {
    'S': [('A', 2), ('B', 2)],
    'A': [('C', 2), ('D', 5)],
    'B': [('D', 2)],
    'C': [('G', 3)],
    'D': [('G', 6)],
    'G': [],
}

heuristica = {
    'S': 7,
    'A': 5,
    'B': 7,
    'C': 3,
    'D': 6,
    'G': 0,
}

INICIO = 'S'
OBJETIVO = 'G'

print('Grafo y heurística cargados.')
pprint(grafo)
pprint(heuristica)

## Limpieza y tratamiento de valores faltantes

Se valida que el grafo y la heurística sean consistentes: no debe haber nodos sin heurística ni referencias a vecinos inexistentes.

In [ ]:
def validar_entrada(grafo, heuristica):
    nodos_grafo = set(grafo)
    nodos_heuristica = set(heuristica)
    vecinos = {vecino for adyacentes in grafo.values() for vecino, _ in adyacentes}

    faltan_en_heuristica = sorted(nodos_grafo - nodos_heuristica)
    faltan_en_grafo = sorted(vecinos - nodos_grafo)

    if faltan_en_heuristica:
        raise ValueError(f'Hay nodos del grafo sin heurística: {faltan_en_heuristica}')
    if faltan_en_grafo:
        raise ValueError(f'Hay vecinos referenciados que no existen en el grafo: {faltan_en_grafo}')

    return True

assert validar_entrada(grafo, heuristica)
print('Validación estructural superada.')

## Preprocesamiento de variables y features

En un problema de búsqueda sobre grafos no hay codificación ni escalado: el preprocesamiento consiste en ordenar la información y preparar estructuras de apoyo para la ejecución.

## Análisis exploratorio de datos

Acá se observan propiedades simples del grafo: cantidad de nodos, aristas, grados de salida y valores de heurística.

In [ ]:
def resumen_grafo(grafo, heuristica):
    nodos = sorted(grafo)
    cantidad_aristas = sum(len(adyacentes) for adyacentes in grafo.values())
    grados = {nodo: len(grafo[nodo]) for nodo in nodos}

    print(f'Nodos: {len(nodos)}')
    print(f'Aristas: {cantidad_aristas}')
    print('Grados de salida:')
    for nodo in nodos:
        print(f'  {nodo}: {grados[nodo]}')

    print('Heurística:')
    for nodo in nodos:
        print(f'  {nodo}: {heuristica[nodo]}')

resumen_grafo(grafo, heuristica)

## División de datos en entrenamiento y prueba

Este problema no usa train/test clásico porque no hay datos supervisados. La búsqueda se ejecuta sobre un único grafo fijo y la comparación se hace entre algoritmos.

In [ ]:
train_set = None
test_set = None
print('No se realiza partición train/test: el TP trabaja con un grafo fijo y una única consulta de búsqueda.')

## Construcción del modelo

Se define el bloque común de búsqueda y las tres variantes: UCS, Voraz y A*.

In [ ]:
def crear_nodo(estado, padre=None, accion=None, g=0, h=0):
    return {
        'estado': estado,
        'padre': padre,
        'accion': accion,
        'g': g,
        'h': h,
        'f': g + h,
    }


class ColaPrioridad:
    def __init__(self):
        self._heap = []
        self._contador = itertools.count()

    def insertar(self, prioridad, nodo):
        contador = next(self._contador)
        heapq.heappush(self._heap, (prioridad, contador, nodo))

    def extraer(self):
        _, _, nodo = heapq.heappop(self._heap)
        return nodo

    def esta_vacia(self):
        return len(self._heap) == 0

    def contenido(self):
        return [(prioridad, nodo['estado']) for prioridad, _, nodo in sorted(self._heap)]

    def __len__(self):
        return len(self._heap)


def _calcular_prioridad(modo, g, h):
    if modo == 'ucs':
        return g
    if modo == 'voraz':
        return h
    if modo == 'a_estrella':
        return g + h
    raise ValueError(f'Modo desconocido: {modo}')


def _reconstruir_camino(nodo):
    camino = []
    actual = nodo
    while actual is not None:
        camino.append(actual['estado'])
        actual = actual['padre']
    camino.reverse()
    return camino


def buscar(modo, grafo, heuristica, inicio, objetivo):
    contador_generados = 0
    contador_expandidos = 0
    frontera_maxima = 0
    reaperturas = 0
    traza = []

    mejor_g = {}
    frontera = ColaPrioridad()

    raiz = crear_nodo(inicio, padre=None, accion=None, g=0, h=heuristica[inicio])
    raiz['f'] = _calcular_prioridad(modo, raiz['g'], raiz['h'])
    mejor_g[inicio] = 0
    frontera.insertar(raiz['f'], raiz)
    contador_generados += 1

    while not frontera.esta_vacia():
        frontera_maxima = max(frontera_maxima, len(frontera))
        nodo = frontera.extraer()
        estado = nodo['estado']

        if nodo['g'] > mejor_g.get(estado, float('inf')):
            continue

        contador_expandidos += 1
        traza.append({
            'expandido': estado,
            'g': nodo['g'],
            'h': nodo['h'],
            'f': nodo['f'],
            'frontera_restante': frontera.contenido(),
        })

        if estado == objetivo:
            return {
                'camino': _reconstruir_camino(nodo),
                'costo': nodo['g'],
                'estados_generados': contador_generados,
                'estados_expandidos': contador_expandidos,
                'frontera_maxima': frontera_maxima,
                'reaperturas': reaperturas,
                'traza': traza,
            }

        for vecino, costo in grafo[estado]:
            nuevo_g = nodo['g'] + costo
            if nuevo_g < mejor_g.get(vecino, float('inf')):
                if vecino in mejor_g:
                    reaperturas += 1
                mejor_g[vecino] = nuevo_g
                h_vecino = heuristica[vecino]
                hijo = crear_nodo(
                    vecino,
                    padre=nodo,
                    accion=f'{estado} -> {vecino}',
                    g=nuevo_g,
                    h=h_vecino,
                )
                hijo['f'] = _calcular_prioridad(modo, nuevo_g, h_vecino)
                frontera.insertar(hijo['f'], hijo)
                contador_generados += 1

    return {
        'camino': None,
        'costo': None,
        'estados_generados': contador_generados,
        'estados_expandidos': contador_expandidos,
        'frontera_maxima': frontera_maxima,
        'reaperturas': reaperturas,
        'traza': traza,
    }


def ucs(grafo, heuristica, inicio, objetivo):
    return buscar('ucs', grafo, heuristica, inicio, objetivo)


def voraz(grafo, heuristica, inicio, objetivo):
    return buscar('voraz', grafo, heuristica, inicio, objetivo)


def a_estrella(grafo, heuristica, inicio, objetivo):
    return buscar('a_estrella', grafo, heuristica, inicio, objetivo)

print('Algoritmos definidos.')

## Entrenamiento del modelo

En este TP, "entrenar" equivale a ejecutar los tres algoritmos sobre el mismo problema para comparar su comportamiento.

In [ ]:
resultado_ucs = ucs(grafo, heuristica, INICIO, OBJETIVO)
resultado_voraz = voraz(grafo, heuristica, INICIO, OBJETIVO)
resultado_a_estrella = a_estrella(grafo, heuristica, INICIO, OBJETIVO)

resultados = {
    'UCS': resultado_ucs,
    'Voraz': resultado_voraz,
    'A*': resultado_a_estrella,
}

print('Ejecución completada.')

## Evaluación del modelo

Se comparan el camino encontrado, el costo total y las métricas internas de cada algoritmo.

In [ ]:
def imprimir_traza(nombre_algoritmo, resultado):
    print(f"\n{'=' * 60}")
    print(f'TRAZA: {nombre_algoritmo}')
    print(f"{'=' * 60}")

    for i, paso in enumerate(resultado['traza'], start=1):
        print(f"\nPaso {i}: se expande '{paso['expandido']}' (g={paso['g']}, h={paso['h']}, f={paso['f']})")
        if paso['frontera_restante']:
            frontera_str = ', '.join(f"{estado}(prio={prioridad})" for prioridad, estado in paso['frontera_restante'])
            print(f'  Frontera restante: [{frontera_str}]')
        else:
            print('  Frontera restante: []')

    camino = ' -> '.join(resultado['camino']) if resultado['camino'] else 'NO ENCONTRADO'
    print(f'\nCamino encontrado: {camino}')
    print(f"Costo total: {resultado['costo']}")


def imprimir_tabla_comparativa(resultados):
    encabezado = (
        f"{'Algoritmo':<12} | {'Camino':<18} | {'Costo':<6} | "
        f"{'Prioridad usada':<16} | {'Generados':<10} | "
        f"{'Expandidos':<11} | {'Frontera máx':<13} | {'Reaperturas':<11}"
    )
    print(f"\n{'=' * len(encabezado)}")
    print('TABLA COMPARATIVA')
    print(f"{'=' * len(encabezado)}")
    print(encabezado)
    print('-' * len(encabezado))

    prioridad_por_algoritmo = {
        'UCS': 'g',
        'Voraz': 'h',
        'A*': 'g + h',
    }

    for nombre, resultado in resultados.items():
        camino_str = ' -> '.join(resultado['camino']) if resultado['camino'] else 'NO ENCONTRADO'
        print(
            f"{nombre:<12} | {camino_str:<18} | {str(resultado['costo']):<6} | "
            f"{prioridad_por_algoritmo.get(nombre, '?'):<16} | "
            f"{resultado['estados_generados']:<10} | "
            f"{resultado['estados_expandidos']:<11} | "
            f"{resultado['frontera_maxima']:<13} | "
            f"{resultado['reaperturas']:<11}"
        )

print('Funciones de reporte definidas.')

## Visualización de resultados

La visualización principal en este TP es textual: trazas de expansión y tabla comparativa final.

In [ ]:
imprimir_traza('UCS', resultado_ucs)
imprimir_traza('Voraz', resultado_voraz)
imprimir_traza('A*', resultado_a_estrella)
imprimir_tabla_comparativa(resultados)

## Guardado de artefactos y resultados

Se guardan las salidas principales en JSON para que puedas reutilizarlas o adjuntarlas junto con la entrega.

In [ ]:
output_dir = Path('/content/tp_busqueda_resultados')
output_dir.mkdir(parents=True, exist_ok=True)

resultados_serializables = {
    nombre: {
        'camino': resultado['camino'],
        'costo': resultado['costo'],
        'estados_generados': resultado['estados_generados'],
        'estados_expandidos': resultado['estados_expandidos'],
        'frontera_maxima': resultado['frontera_maxima'],
        'reaperturas': resultado['reaperturas'],
    }
    for nombre, resultado in resultados.items()
}

with (output_dir / 'resultados_busqueda.json').open('w', encoding='utf-8') as archivo:
    json.dump(resultados_serializables, archivo, ensure_ascii=False, indent=2)

print(f'Resultados guardados en: {output_dir / "resultados_busqueda.json"}')

## Exportación del notebook para entrega

El notebook quedó preparado para abrirse en Colab, ejecutarse de principio a fin y descargarse desde la interfaz de Google Colab cuando lo necesites.

## Respuestas de análisis

### ¿Por qué Voraz y A* coinciden en este grafo particular?

Voraz y A* encontraron el mismo camino porque, en este grafo, la heurística ayudó a elegir correctamente los nodos. Desde A, Voraz eligió C porque tenía el menor valor de h (h(C)=3). A* también eligió C porque al combinar el costo recorrido con la heurística obtuvo un valor favorable: f(C)=g(C)+h(C)=4+3=7. Como C realmente conducía al camino más barato, ambos algoritmos terminaron encontrando S → A → C → G, con costo 7.

Esto ocurre en este grafo particular y no significa que Voraz y A* siempre encuentren el mismo camino.

### ¿Voraz garantiza la solución óptima en general?

No. Voraz solamente tiene en cuenta la heurística h(n), es decir, qué tan cerca parece estar un nodo del objetivo. No tiene en cuenta cuánto costó llegar hasta ese nodo.

Por ejemplo, podría elegir un nodo que parece muy cercano al objetivo, pero al que se llegó mediante un camino muy costoso. Por eso Voraz puede encontrar una solución rápidamente, pero no garantiza que sea la solución más barata.

En este ejercicio encontró el camino óptimo de costo 7, pero esto no está garantizado en general.

### ¿Qué pasa con A* si h(n) = 0 para todos los nodos?

A* utiliza la función:

f(n) = g(n) + h(n)

Si h(n)=0 para todos los nodos, entonces:

f(n) = g(n)

Por lo tanto, A* pasa a utilizar solamente el costo acumulado desde el inicio, exactamente igual que UCS. Por eso se dice que UCS es un caso particular de A* cuando la heurística es cero.

### ¿Hubo reaperturas? ¿Por qué?

Sí. En UCS hubo una reapertura del nodo D.

Primero se encontró D mediante S → A → D, con un costo de 7. Después se encontró otra forma de llegar a D mediante S → B → D, con un costo de 4.

Como 4 es menor que 7, se actualizó el costo de D y se volvió a considerar ese nodo. Por eso se registró una reapertura.

En Voraz y A* no hubo reaperturas en esta ejecución.

### ¿"Expandir menos nodos" es lo mismo que "encontrar la solución más barata"?

No. Son dos conceptos diferentes.

Expandir menos nodos significa que el algoritmo realizó menos trabajo durante la búsqueda. Encontrar la solución más barata significa que encontró el camino de menor costo.

En este ejercicio, UCS expandió 6 nodos, mientras que Voraz y A* expandieron 4. Sin embargo, los tres encontraron el mismo camino S → A → C → G, con costo 7.

Por lo tanto, en este caso Voraz y A* fueron más eficientes que UCS, pero esto no significa que expandir menos nodos garantice encontrar una solución más barata. Voraz, por ejemplo, puede expandir pocos nodos y aun así encontrar una solución que no sea óptima.